## **Required Imports**

In [2]:
import os
import pandas as pd
import numpy as np

import warnings

In [3]:
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

## **Dataset Loading**

In [4]:
INPUT_DIR  = "data"         
OUTPUT_DIR = "cleaned" 
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
players      = pd.read_csv(f"{INPUT_DIR}/players.csv",           low_memory=False)
transfers    = pd.read_csv(f"{INPUT_DIR}/transfers.csv",         low_memory=False)
valuations   = pd.read_csv(f"{INPUT_DIR}/player_valuations.csv", low_memory=False)
clubs        = pd.read_csv(f"{INPUT_DIR}/clubs.csv",             low_memory=False)
appearances  = pd.read_csv(f"{INPUT_DIR}/appearances.csv",       low_memory=False)

## **Tables Summary Report**

### **Players Table**

In [6]:
print(f'Players Table Shape: {players.shape[0]:,} rows x {players.shape[1]} columns')

column_report = pd.DataFrame({
    'dtype'        : players.dtypes,
    'nulls'        : players.isna().sum(),
    'null_%'       : (players.isna().mean() * 100).round(2),
    'unique'       : players.nunique(),
    'min_value'   : players.min(numeric_only=True),
    'max_value'   : players.max(numeric_only=True),
})
column_report

Players Table Shape: 47,702 rows x 26 columns


,dtype,nulls,null_%,unique,min_value,max_value
agent_name,object,22180,46.50,3826,NaN,NaN
city_of_birth,object,4887,10.24,11651,NaN,NaN
contract_expiration_date,object,16488,34.56,236,NaN,NaN
country_of_birth,object,5162,10.82,203,NaN,NaN
country_of_citizenship,object,271,0.57,198,NaN,NaN
current_club_domestic_competition_id,object,2713,5.69,32,NaN,NaN
current_club_id,int64,0,0.00,1751,2.0,138189.0
current_club_name,object,2713,5.69,793,NaN,NaN
current_national_team_id,float64,44611,93.52,116,3262.0,53982.0
date_of_birth,object,49,0.10,10166,NaN,NaN


In [7]:
players.head(5)

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,IFM,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000.0,3000000.0


In [8]:
full_row_dupes = players[players.duplicated()]
print(f'Full-Row Duplicates: {len(full_row_dupes)}')

player_id_dupes = players[players.duplicated(subset='player_id', keep=False)]
print(f'Player ID Duplicates: {len(player_id_dupes)}')

name_dob_dupes = players[players.duplicated(subset=['name', 'date_of_birth'], keep=False)]
print(f'Name + Date of Birth Duplicates: {len(name_dob_dupes)}')
name_dob_dupes.sort_values(['name', 'date_of_birth']).head(10)

Full-Row Duplicates: 0
Player ID Duplicates: 0
Name + Date of Birth Duplicates: 2


,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
23679,398057,João,Ricciulli,João Ricciulli,2018,1038,joao-ricciulli,Guinea-Bissau,Bissau,Guinea-Bissau,...,Evlon Sports & Mgmt,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/joao-ricciulli...,IT1,UC Sampdoria,25000.0,100000.0
26162,461923,João,Ricciulli,João Ricciulli,2022,2420,joao-ricciulli,NaN,Bissau,NaN,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/joao-ricciulli...,PO1,Vitória Sport Clube,100000.0,100000.0


### **Clubs Table**

In [9]:
print(f'Clubs Table Shape: {clubs.shape[0]:,} rows x {clubs.shape[1]} columns')

column_report = pd.DataFrame({
    'dtype'        : clubs.dtypes,
    'nulls'        : clubs.isna().sum(),
    'null_%'       : (clubs.isna().mean() * 100).round(2),
    'unique'       : clubs.nunique(),
    'min_value'   : clubs.min(numeric_only=True),
    'max_value'   : clubs.max(numeric_only=True),
})
column_report

Clubs Table Shape: 796 rows x 17 columns


,dtype,nulls,null_%,unique,min_value,max_value
average_age,float64,38,4.77,83,0.0,30.9
club_code,object,0,0.00,796,NaN,NaN
club_id,int64,0,0.00,796,3.0,118373.0
coach_name,object,706,88.69,90,NaN,NaN
domestic_competition_id,object,0,0.00,32,NaN,NaN
filename,object,0,0.00,14,NaN,NaN
foreigners_number,int64,0,0.00,29,0.0,28.0
foreigners_percentage,float64,57,7.16,247,2.4,100.0
last_season,int64,0,0.00,14,2012.0,2025.0
name,object,0,0.00,796,NaN,NaN


In [10]:
clubs.head(5)

,club_id,club_code,name,domestic_competition_id,total_market_value,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,net_transfer_record,coach_name,last_season,filename,url
0,10,arminia-bielefeld,Arminia Bielefeld,L1,NaN,27,25.3,15,55.6,4,SchücoArena,26515,+€5.90m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/arminia-bielef...
1,10004,paris-fc,Paris Football Club,FR1,NaN,31,28.5,17,54.8,8,Stade Jean Bouin,19904,€-72.30m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/paris-fc/start...
2,10010,esporte-clube-bahia,Esporte Clube Bahia,BRA1,NaN,32,26.2,6,18.8,3,Arena Fonte Nova,47364,+€8.14m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/esporte-clube-...
3,1003,leicester-city,Leicester City,GB1,NaN,29,25.9,17,58.6,9,King Power Stadium,32259,+€57.30m,Steve Cooper,2024,../data/raw/transfermarkt-scraper/2024/clubs.j...,https://www.transfermarkt.co.uk/leicester-city...
4,1005,us-lecce,Unione Sportiva Lecce,IT1,NaN,27,25.1,23,85.2,10,Ettore Giardiniero,31559,+€8.62m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/us-lecce/start...


In [11]:
full_row_dupes = clubs[clubs.duplicated()]
print(f'Full-Row Duplicates: {len(full_row_dupes)}')

id_dupes = clubs[clubs.duplicated(subset='club_id', keep=False)]
print(f'Club ID Duplicates: {len(id_dupes)}')

code_dupes = clubs[clubs.duplicated(subset='club_code', keep=False)]
print(f'Club Code Duplicates: {len(code_dupes)}')   

Full-Row Duplicates: 0
Club ID Duplicates: 0
Club Code Duplicates: 0


### **Transfers Table**

In [12]:
print(f'Transfers Table Shape: {transfers.shape[0]:,} rows x {transfers.shape[1]} columns')

column_report = pd.DataFrame({
    'dtype'        : transfers.dtypes,
    'nulls'        : transfers.isna().sum(),
    'null_%'       : (transfers.isna().mean() * 100).round(2),
    'unique'       : transfers.nunique(),
    'min_value'   : transfers.min(numeric_only=True),
    'max_value'   : transfers.max(numeric_only=True),
})
column_report

Transfers Table Shape: 157,186 rows x 10 columns


,dtype,nulls,null_%,unique,min_value,max_value
from_club_id,int64,0,0.00,15288,1.0,140641.0
from_club_name,object,0,0.00,16078,NaN,NaN
market_value_in_eur,float64,60848,38.71,191,10000.0,180000000.0
player_id,int64,0,0.00,20855,3333.0,1529719.0
player_name,object,0,0.00,20546,NaN,NaN
to_club_id,int64,0,0.00,11521,1.0,140573.0
to_club_name,object,0,0.00,12302,NaN,NaN
transfer_date,object,0,0.00,5412,NaN,NaN
transfer_fee,float64,54833,34.88,1449,0.0,222000000.0
transfer_season,object,0,0.00,35,NaN,NaN


In [13]:
transfers.head(5)

,player_id,transfer_date,transfer_season,from_club_id,to_club_id,from_club_name,to_club_name,transfer_fee,market_value_in_eur,player_name
0,467994,2030-06-30,25/26,5621,749,Reggiana,FC Empoli,0.0,700000.0,Luca Belardinelli
1,784335,2027-07-18,27/28,6505,6502,Gimcheon Sangmu,Jeonbuk Hyundai,0.0,500000.0,Jun-soo Byeon
2,402135,2027-07-04,27/28,6505,515,Gimcheon Sangmu,Without Club,NaN,350000.0,Jun-su Ahn
3,716435,2027-07-04,27/28,6505,30925,Gimcheon Sangmu,Gwangju FC,0.0,325000.0,Kang-hyun Lee
4,803933,2027-06-30,26/27,4467,14,TSV Hartberg,Austria Vienna,0.0,300000.0,Luca Pazourek


In [14]:
full_dupes = transfers[transfers.duplicated()]
print(f'Full-Row Duplicates: {len(full_dupes)}')

transfer_key = ['player_id', 'transfer_date', 'from_club_id', 'to_club_id']
transfer_dupes = transfers[transfers.duplicated(subset=transfer_key, keep=False)]
print(f'Transfer-Key Duplicates: {len(transfer_dupes)}')
if len(transfer_dupes) > 0:
    display(transfer_dupes.sort_values(transfer_key).head(10))

Full-Row Duplicates: 0
Transfer-Key Duplicates: 0


### **Player Valuations Table**

In [15]:
print(f'Valuations Table Shape: {valuations.shape[0]:,} rows x {valuations.shape[1]} columns')

column_report = pd.DataFrame({
    'dtype'        : valuations.dtypes,
    'nulls'        : valuations.isna().sum(),
    'null_%'       : (valuations.isna().mean() * 100).round(2),
    'unique'       : valuations.nunique(),
    'min_value'   : valuations.min(numeric_only=True),
    'max_value'   : valuations.max(numeric_only=True),
})
column_report

Valuations Table Shape: 616,377 rows x 6 columns


,dtype,nulls,null_%,unique,min_value,max_value
current_club_id,float64,24,0.00,4676,1.0,133929.0
current_club_name,object,0,0.00,7488,NaN,NaN
date,object,0,0.00,5657,NaN,NaN
market_value_in_eur,int64,0,0.00,425,10000.0,200000000.0
player_club_domestic_competition_id,object,71796,11.65,32,NaN,NaN
player_id,int64,0,0.00,39361,10.0,1519157.0


In [16]:
valuations.head(5)

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057.0,BE1
1,342216,2001-07-20,100000,Unknown,1241.0,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126.0,TR1
3,6893,2003-12-15,900000,Galatasaray,984.0,GB1
4,10,2004-10-04,7000000,SV Werder Bremen,398.0,IT1


In [17]:
full_dupes = valuations[valuations.duplicated()]
print(f'Full-Row Duplicates: {len(full_dupes)}')

valuation_key = ['player_id', 'date', 'current_club_id']
valuation_dupes = valuations[valuations.duplicated(subset=valuation_key, keep=False)]
print(f'Valuation-Key Duplicates: {len(valuation_dupes)}')
if len(valuation_dupes) > 0:
    display(valuation_dupes.sort_values(valuation_key).head(10))

Full-Row Duplicates: 0
Valuation-Key Duplicates: 0


### **Appearnaces Table**

In [18]:
print(f'Appearances Table Shape: {appearances.shape[0]:,} rows x {appearances.shape[1]} columns')

column_report = pd.DataFrame({
    'dtype'        : appearances.dtypes,
    'nulls'        : appearances.isna().sum(),
    'null_%'       : (appearances.isna().mean() * 100).round(2),
    'unique'       : appearances.nunique(),
    'min_value'   : appearances.min(numeric_only=True),
    'max_value'   : appearances.max(numeric_only=True),
})
column_report

Appearances Table Shape: 1,862,208 rows x 13 columns


,dtype,nulls,null_%,unique,min_value,max_value
appearance_id,object,0,0.0,1862208,NaN,NaN
assists,int64,0,0.0,7,0.0,6.0
competition_id,object,0,0.0,44,NaN,NaN
date,object,0,0.0,4039,NaN,NaN
game_id,int64,0,0.0,72482,2211607.0,4839901.0
goals,int64,0,0.0,7,0.0,6.0
minutes_played,int64,0,0.0,122,1.0,148.0
player_club_id,int64,0,0.0,1180,1.0,132877.0
player_current_club_id,int64,0,0.0,1100,-1.0,138189.0
player_id,int64,0,0.0,28665,10.0,1510255.0


In [19]:
appearances.head(5)

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45
3,2234418_73333,2234418,73333,1274,76,2012-07-05,Vegar Hedenstad,ELQ,0,0,0,0,90
4,2234421_122011,2234421,122011,195,3008,2012-07-05,Markus Henriksen,ELQ,0,0,0,1,90


In [22]:
full_dupes = appearances[appearances.duplicated()]
print(f'Full-Row Duplicates: {len(full_dupes)}')

id_dupes = appearances[appearances.duplicated(subset='appearance_id', keep=False)]
print(f'Appearance ID Duplicates: {len(id_dupes)}')

appearance_key = ['game_id', 'player_id']
appearance_dupes = appearances[appearances.duplicated(subset=appearance_key, keep=False)]
print(f'Appearance-Key Duplicates: {len(appearance_dupes)}')
if len(appearance_dupes) > 0:
    display(appearance_dupes.sort_values(appearance_key).head(10))

Full-Row Duplicates: 0
Appearance ID Duplicates: 0
Appearance-Key Duplicates: 0


## **Data Cleaning**

### **Players Table**

**Dropping non-analytical and 90% null columns**

In [19]:
players.drop(columns=["current_national_team_id"], inplace=True)

players.drop(columns=["image_url", "url"], inplace=True)

**Datetime columns formatting**

In [20]:
players["date_of_birth"] = pd.to_datetime(players["date_of_birth"], errors="coerce").dt.date
players["contract_expiration_date"] = pd.to_datetime(players["contract_expiration_date"], errors="coerce").dt.date

**Numeric columns formatting**

In [21]:
numeric_cols = ['height_in_cm', 'international_caps', 'international_goals', 'market_value_in_eur', 'highest_market_value_in_eur']
for col in numeric_cols: 
    players[col] = pd.to_numeric(players[col], errors='coerce')

**Height range correction**

In [22]:
print(f"Players with Height < 150 cm: {players[players['height_in_cm'] < 150].shape[0]}")
print("\n")

players.loc[players["height_in_cm"] < 150, "height_in_cm"] = np.nan
players["height_in_cm"] = players.groupby("position")["height_in_cm"].transform(lambda x: x.fillna(x.median()))

for position in players["position"].unique():
    print(f"{position} Position (Median = {players[players['position'] == position]['height_in_cm'].median()}): {players[players['position'] == position]['height_in_cm'].min()} - {players[players['position'] == position]['height_in_cm'].max()}")

Players with Height < 150 cm: 14


Attack Position (Median = 180.0): 155.0 - 204.0
Goalkeeper Position (Median = 189.0): 172.0 - 210.0
Defender Position (Median = 184.0): 161.0 - 206.0
Midfield Position (Median = 179.0): 155.0 - 202.0
Missing Position (Median = 178.0): 165.0 - 196.0


**Null first name handling**

In [23]:
print(f"Null count for first_name before processing: {players['first_name'].isna().sum()}")

no_first_name = players['first_name'].isna()

# case of name being a single word 
single_name = no_first_name & (players['name'].str.strip().str.split().str.len() == 1)
players.loc[single_name, 'first_name'] = ''

# case of name being multiple words (infer first_name)
multi_name = no_first_name & (players['name'].str.strip().str.split().str.len() > 1)
players.loc[multi_name, 'first_name'] = (
    players.loc[multi_name, 'name'].str.strip().str.split().str[:-1].str.join(' ')
)

print(f'\nRemaining first_name nulls: {players["first_name"].isna().sum()}')
print(f'Single-name players flagged: {single_name.sum()}')
print(f'Multi-name players (inferred first_name from name): {multi_name.sum()}')

Null count for first_name before processing: 3095

Remaining first_name nulls: 0
Single-name players flagged: 1713
Multi-name players (inferred first_name from name): 1382


**Logical consistency check for name columns**

In [24]:
print("Checking consistency between name, first_name, and last_name columns.")

consistency_issues = players[
    ~players.apply(
        lambda row: (
            (pd.isna(row['first_name']) or row['first_name'] == '' or row['first_name'] in str(row['name']))
            and
            # last name check
            (pd.isna(row['last_name']) or row['last_name'] == '' or row['last_name'] in str(row['name']))
        ),
        axis=1
    )
]

if consistency_issues.empty:
    print("No consistency issues found - All names are valid.")
else:
    print("Found inconsistencies:")
    print(consistency_issues)

Checking consistency between name, first_name, and last_name columns.
No consistency issues found - All names are valid.


**Null country columns handling**

In [25]:
print(f'Null count for country_of_birth: {players["country_of_birth"].isna().sum()}')
print(f'Null count for country_of_citizenship: {players["country_of_citizenship"].isna().sum()}')

cob_mask = players['country_of_birth'].isna() & players['country_of_citizenship'].notna()
players.loc[cob_mask, 'country_of_birth'] = players.loc[cob_mask, 'country_of_citizenship']

coc_mask = players['country_of_citizenship'].isna() & players['country_of_birth'].notna()
players.loc[coc_mask, 'country_of_citizenship'] = players.loc[coc_mask, 'country_of_birth']

print(f'\nRemaining nulls in country_of_birth after inference: {players["country_of_birth"].isna().sum()}')
print(f'Remaining nulls in country_of_citizenship after inference: {players["country_of_citizenship"].isna().sum()}')

players['country_of_birth'].fillna('Unknown', inplace=True)
players['country_of_citizenship'].fillna('Unknown', inplace=True)

print(f'\nRemaining nulls in country_of_birth: {players["country_of_birth"].isna().sum()}')
print(f'Remaining nulls in country_of_citizenship: {players["country_of_citizenship"].isna().sum()}')

Null count for country_of_birth: 5162
Null count for country_of_citizenship: 271

Remaining nulls in country_of_birth after inference: 270
Remaining nulls in country_of_citizenship after inference: 270

Remaining nulls in country_of_birth: 0
Remaining nulls in country_of_citizenship: 0


**Null date values (DoB and Contract Expiration) handling**

In [26]:
players.loc[players["date_of_birth"].isna(), "date_of_birth"] = pd.NaT
players.loc[players["contract_expiration_date"].isna(), "contract_expiration_date"] = pd.NaT

players['dob_missing_flag'] = players['date_of_birth'].isna().astype(int)
print(f'NaT count for date_of_birth: {players["date_of_birth"].isna().sum()}')
print(f'Rows flagged with dob_missing_flag = 1: {players["dob_missing_flag"].sum()}')

players["has_contract"] = players["contract_expiration_date"].notna().astype(int)
contract_counts = players["has_contract"].value_counts().reset_index()
contract_counts.columns = ["has_contract", "count"]
print(f"{'\nHas Contract (contract_expiration_date = NaT)':<45} {'Count':>10}")
for _, row in contract_counts.iterrows():
    print(f"{row['has_contract']:<45} {row['count']:>10,}")
    
today = pd.Timestamp('today').date()

active = players[
    players['contract_expiration_date'].notna() &
    (
        pd.to_datetime(players['contract_expiration_date'], errors='coerce').dt.date >= today
    )
]

expired = players[
    players['contract_expiration_date'].notna() &
    (
        pd.to_datetime(players['contract_expiration_date'], errors='coerce').dt.date < today
    )
]
print(f'\nPlayers with expired contracts still in dataset: {len(expired)}')
print(f'Players with active contracts: {len(active)}')

NaT count for date_of_birth: 49
Rows flagged with dob_missing_flag = 1: 49

Has Contract (contract_expiration_date = NaT)      Count
1                                                 31,214
0                                                 16,488

Players with expired contracts still in dataset: 10043
Players with active contracts: 21171


**Position and sub position column handling**

In [27]:
pos_missing = (players["position"] == "Missing").sum()
players["position"] = players["position"].replace("Missing", np.nan).fillna("Unknown")
print(f"position: {pos_missing} 'Missing' strings converted to 'Unknown'")

position: 455 'Missing' strings converted to 'Unknown'


In [28]:
print(f'Null count for sub_position: {players["sub_position"].isna().sum()}')

print(players.groupby('position')['sub_position'].value_counts())

mode_by_position = (
    players.dropna(subset=['sub_position'])
    .groupby('position')['sub_position']
    .agg(lambda x: x.mode()[0])
)
print('\nMode sub_position per position:')
print(mode_by_position)

def fill_sub_position(row):
    if pd.isna(row['sub_position']):
        return mode_by_position.get(row['position'], np.nan)
    return row['sub_position']

players['sub_position'] = players.apply(fill_sub_position, axis=1)

print(f'\nRemaining nulls in sub_position after filling: {players["sub_position"].isna().sum()}')

if players['sub_position'].isna().sum() > 0:
    players['sub_position'].fillna('Unknown', inplace=True)

print(f'\nRemaining nulls in sub_position: {players["sub_position"].isna().sum()}')

Null count for sub_position: 455
position    sub_position      
Attack      Centre-Forward        6464
            Left Winger           3114
            Right Winger          3021
            Second Striker         278
Defender    Centre-Back           8336
            Right-Back            3523
            Left-Back             3346
Goalkeeper  Goalkeeper            5529
Midfield    Central Midfield      5220
            Defensive Midfield    3959
            Attacking Midfield    3355
            Right Midfield         554
            Left Midfield          548
Name: count, dtype: int64

Mode sub_position per position:
position
Attack          Centre-Forward
Defender           Centre-Back
Goalkeeper          Goalkeeper
Midfield      Central Midfield
Name: sub_position, dtype: object

Remaining nulls in sub_position after filling: 455

Remaining nulls in sub_position: 0


**Null foot values handling**

In [29]:
print('Overall foot distribution:')
print(players['foot'].value_counts(dropna=False))

print('\nFoot distribution by position:')
print(players.groupby('position')['foot'].value_counts(normalize=True).round(3))

Overall foot distribution:
foot
right    30106
left     10578
NaN       5256
both      1762
Name: count, dtype: int64

Foot distribution by position:
position    foot 
Attack      right    0.726
            left     0.219
            both     0.056
Defender    right    0.625
            left     0.347
            both     0.029
Goalkeeper  right    0.847
            left     0.129
            both     0.024
Midfield    right    0.740
            left     0.211
            both     0.049
Unknown     right    0.657
            left     0.269
            both     0.074
Name: proportion, dtype: float64


In [30]:
mode_foot_by_position = (
    players.dropna(subset=['foot'])
    .groupby('position')['foot']
    .agg(lambda x: x.mode()[0])
)
print('Mode foot per position:', mode_foot_by_position.to_dict())

mask_foot = players['foot'].isna()
players.loc[mask_foot, 'foot'] = players.loc[mask_foot, 'position'].map(mode_foot_by_position)
players['foot'] = players['foot'].astype('category')

print(f'\nRemaining nulls in foot: {players["foot"].isna().sum()}')

Mode foot per position: {'Attack': 'right', 'Defender': 'right', 'Goalkeeper': 'right', 'Midfield': 'right', 'Unknown': 'right'}

Remaining nulls in foot: 0


**Null agent name handling**

In [31]:
players['agent_name'] = players['agent_name'].fillna('Unknown')

**International caps and goals column handling**

In [32]:
players['international_caps']  = players['international_caps'].fillna(0).astype(int)
players['international_goals'] = players['international_goals'].fillna(0).astype(int)

# sanity check: goals can never be more than 3x caps
invalid_goal_cap = players[players['international_goals'] > (3 * players['international_caps'])]
print(f'Rows where goals > 3x caps: {len(invalid_goal_cap)}')
if len(invalid_goal_cap) > 0:
    display(invalid_goal_cap[['player_id','name','international_caps','international_goals']])

Rows where goals > 3x caps: 0


**Current club values null handling**

In [33]:
print(f'Null count for current_club_name: {players["current_club_name"].isna().sum()}')
print(f'Null count for current_club_domestic_competition_id: {players["current_club_domestic_competition_id"].isna().sum()}') 

clubs_lookup = clubs[['club_id', 'name', 'domestic_competition_id']].copy()
clubs_lookup.columns = ['current_club_id', 'club_name_lookup', 'comp_id_lookup']

merged = players.merge(clubs_lookup, on='current_club_id', how='left')

mask_club = players['current_club_name'].isna()
players.loc[mask_club, 'current_club_name'] = merged.loc[mask_club, 'club_name_lookup']

mask_comp = players['current_club_domestic_competition_id'].isna()
players.loc[mask_comp, 'current_club_domestic_competition_id'] = merged.loc[mask_comp, 'comp_id_lookup']

print(f'\nRemaining nulls in current_club_name after filling: {players["current_club_name"].isna().sum()}')
print(f'Remaining nulls in current_club_domestic_competition_id after filling: {players["current_club_domestic_competition_id"].isna().sum()}')

Null count for current_club_name: 2713
Null count for current_club_domestic_competition_id: 2713

Remaining nulls in current_club_name after filling: 2713
Remaining nulls in current_club_domestic_competition_id after filling: 2713


In [34]:
unmatched_club_ids = players.loc[mask_club, 'current_club_id'].unique()

print(f'\nUnique current_club_id values that could not be matched to clubs table:')
unmatched_club_ids = sorted(unmatched_club_ids)
print(unmatched_club_ids)
print(f'Number of unmatched current_club_id values: {len(unmatched_club_ids)}')

players.loc[mask_club, 'current_club_id'] = players.loc[mask_club, 'current_club_id'].where(
    ~players.loc[mask_club, 'current_club_id'].isin(unmatched_club_ids), np.nan
)

players.loc[mask_club, 'current_club_name'] = players.loc[mask_club, 'current_club_name'].where(
    ~players.loc[mask_club, 'current_club_id'].isin(unmatched_club_ids), 'Unknown'
)

players.loc[mask_club, 'current_club_domestic_competition_id'] = players.loc[
    mask_club, 'current_club_domestic_competition_id'
].where(
    ~players.loc[mask_club, 'current_club_id'].isin(unmatched_club_ids), np.nan
)

unmatched_club_ids = players.loc[mask_club, 'current_club_id'].unique()

print(f'\nUnique current_club_id values that could not be matched to clubs table:')
unmatched_club_ids = sorted(unmatched_club_ids)
print(unmatched_club_ids)
print(f'Number of unmatched current_club_id values: {len(unmatched_club_ids)}')


Unique current_club_id values that could not be matched to clubs table:
[np.int64(2), np.int64(7), np.int64(17), np.int64(22), np.int64(45), np.int64(48), np.int64(64), np.int64(71), np.int64(75), np.int64(77), np.int64(84), np.int64(85), np.int64(91), np.int64(93), np.int64(119), np.int64(123), np.int64(156), np.int64(161), np.int64(164), np.int64(184), np.int64(187), np.int64(208), np.int64(220), np.int64(250), np.int64(253), np.int64(279), np.int64(282), np.int64(329), np.int64(332), np.int64(337), np.int64(349), np.int64(352), np.int64(355), np.int64(358), np.int64(392), np.int64(429), np.int64(438), np.int64(440), np.int64(466), np.int64(494), np.int64(500), np.int64(502), np.int64(515), np.int64(518), np.int64(531), np.int64(540), np.int64(554), np.int64(568), np.int64(581), np.int64(588), np.int64(602), np.int64(629), np.int64(630), np.int64(634), np.int64(656), np.int64(657), np.int64(663), np.int64(664), np.int64(675), np.int64(698), np.int64(708), np.int64(710), np.int64(743

**Market value values null handling**

In [35]:
players['age'] = (
    (pd.Timestamp.now().date() - pd.to_datetime(players['date_of_birth'], errors='coerce').dt.date)
    .apply(lambda x: x.days if pd.notnull(x) else None) / 365.25
).round(1)

bins   = [0, 20, 25, 30, 35, 40, 100]
labels = ['<20', '20-24', '25-29', '30-34', '35-39', '40+']
players['age_band'] = pd.cut(players['age'], bins=bins, labels=labels, right=False)

In [36]:
# sanity check: market_value should be <= highest_market_value
inconsistent_mv = players[
    players['market_value_in_eur'].notna() &
    players['highest_market_value_in_eur'].notna() &
    (players['market_value_in_eur'] > players['highest_market_value_in_eur'])
]

print(f'Rows where market_value > highest_market_value: {len(inconsistent_mv)}')
swap_mask = inconsistent_mv.index
if len(swap_mask) > 0:
    display(inconsistent_mv[['player_id','name','market_value_in_eur','highest_market_value_in_eur']])
    players.loc[swap_mask, ['market_value_in_eur', 'highest_market_value_in_eur']] = (
        players.loc[swap_mask, ['highest_market_value_in_eur', 'market_value_in_eur']].values
    )

Rows where market_value > highest_market_value: 0


In [37]:
group_cols = ['position', 'sub_position', 'age_band']

median_mv = (
    players.dropna(subset=['market_value_in_eur'])
    .groupby(group_cols, observed=True)['market_value_in_eur']
    .median()
)

median_mv_fallback = (
    players.dropna(subset=['market_value_in_eur'])
    .groupby(['position', 'age_band'], observed=True)['market_value_in_eur']
    .median()
)

def fill_market_value(row):
    if pd.notna(row['market_value_in_eur']):
        return row['market_value_in_eur']
    key = (row['position'], row['sub_position'], row['age_band'])
    if key in median_mv:
        return median_mv[key]
    key2 = (row['position'], row['age_band'])
    return median_mv_fallback.get(key2, players['market_value_in_eur'].median())

print(f'Nulls before filling market_value_in_eur: {players["market_value_in_eur"].isna().sum()}')
players['market_value_in_eur'] = players.apply(fill_market_value, axis=1)
print(f'Remaining nulls for market_value_in_eur: {players["market_value_in_eur"].isna().sum()}')

Nulls before filling market_value_in_eur: 8476
Remaining nulls for market_value_in_eur: 0


In [38]:
median_hmv = (
    players.dropna(subset=['highest_market_value_in_eur'])
    .groupby(group_cols, observed=True)['highest_market_value_in_eur']
    .median()
)
median_hmv_fallback = (
    players.dropna(subset=['highest_market_value_in_eur'])
    .groupby(['position', 'age_band'], observed=True)['highest_market_value_in_eur']
    .median()
)

def fill_highest_mv(row):
    if pd.notna(row['highest_market_value_in_eur']):
        return row['highest_market_value_in_eur']
    key = (row['position'], row['sub_position'], row['age_band'])
    if key in median_hmv:
        return median_hmv[key]
    key2 = (row['position'], row['age_band'])
    return median_hmv_fallback.get(key2, players['highest_market_value_in_eur'].median())

print(f'Nulls before filling highest_market_value_in_eur: {players["highest_market_value_in_eur"].isna().sum()}')
players['highest_market_value_in_eur'] = players.apply(fill_highest_mv, axis=1)
print(f'Remaining nulls for highest_market_value_in_eur: {players["highest_market_value_in_eur"].isna().sum()}')

Nulls before filling highest_market_value_in_eur: 8476
Remaining nulls for highest_market_value_in_eur: 0


### **Clubs Table**

**Dropping > 70% null and non-analytical columns**

In [39]:
# 100% null
clubs.drop(columns=['total_market_value', 'coach_name'], inplace=True)

clubs.drop(columns=['filename', 'url'], inplace=True)

**Numeric columns formatting**

In [40]:
numeric_cols = ['squad_size', 'average_age', 'foreigners_number',
                'foreigners_percentage', 'national_team_players', 'stadium_seats']
for col in numeric_cols: 
    clubs[col] = pd.to_numeric(clubs[col], errors='coerce')

**Net transfer record parsing**

In [41]:
def parse_transfer_record(value):
    if pd.isna(value) or str(value).strip() == '':
        return np.nan
    
    string = str(value).strip().lower()
    
    string = string.replace('€', '')
    string = string.replace(',', '')
    
    string = string.replace('+-', '-')
    string = string.replace('+', '')
    
    string = string.strip()
    
    try:
        if string in ['0', '-0']:
            return 0.0
        elif string.endswith('bn'):
            return float(string[:-2]) * 1_000_000_000
        elif string.endswith('m'):
            return float(string[:-1]) * 1_000_000
        elif string.endswith('k'):
            return float(string[:-1]) * 1_000
        else:
            return float(string)
    
    except ValueError:
        return np.nan

clubs['net_transfer_record_eur'] = clubs['net_transfer_record'].apply(parse_transfer_record)

In [42]:
failed = clubs[clubs['net_transfer_record_eur'].isna() & clubs['net_transfer_record'].notna()]

print(f'Rows where parsing failed: {len(failed)}')
if len(failed) > 0:
    print('Unparseable Values:')
    print(failed['net_transfer_record'].unique())

print(f'\nValues range after parsing: {clubs["net_transfer_record_eur"].min():,.0f} to {clubs["net_transfer_record_eur"].max():,.0f}')
print(f'Remaining nulls for net_transfer_record_eur: {clubs["net_transfer_record_eur"].isna().sum()}')

clubs.drop(columns=['net_transfer_record'], inplace=True)

Rows where parsing failed: 0

Values range after parsing: -280,300,000 to 135,570,000
Remaining nulls for net_transfer_record_eur: 0


**Average age null and outlier value handling**

In [43]:
print(f'Average Age Distribution:')
print(clubs['average_age'].describe())

invalid_age = clubs[clubs['average_age'] < 17]
print(f'\nOutlier age rows with average_age < 17: {len(invalid_age)}')
if len(invalid_age) > 0:
    display(invalid_age)

Average Age Distribution:
count    758.000000
mean      25.684828
std        2.436554
min        0.000000
25%       24.800000
50%       25.900000
75%       26.900000
max       30.900000
Name: average_age, dtype: float64

Outlier age rows with average_age < 17: 4


,club_id,club_code,name,domestic_competition_id,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,last_season,net_transfer_record_eur
68,11380,scm-gloria-buzau,ASFC Buzau (2016-2025),RO1,0,0.0,0,NaN,0,Gloria,12336,2024,0.0
440,3719,fk-khimki,FC Khimki (-2025),RU1,0,0.0,0,NaN,0,Arena Khimki,18636,2024,0.0
604,60551,sk-dnipro-1,SC Dnipro-1,UKR1,0,0.0,0,NaN,0,Dnipro-Arena,31003,2023,6900000.0
678,71084,western-united-fc,Western United FC,AUS1,0,0.0,0,NaN,0,Ironbark Fields,5000,2024,0.0


In [44]:
clubs.loc[clubs['average_age'] < 17, 'average_age'] = np.nan
print(f'Nulls for average_age after cleaning: {clubs["average_age"].isna().sum()}')

median_age_by_comp = (
    clubs.dropna(subset=['average_age'])
    .groupby('domestic_competition_id')['average_age']
    .median()
)
print('\nMedian average_age per competition (sample):')
print(median_age_by_comp.sort_values().head(10))

# fallback
overall_age_median = clubs['average_age'].median()
print(f'\nOverall median fallback: {overall_age_median}')

Nulls for average_age after cleaning: 42

Median average_age per competition (sample):
domestic_competition_id
BE1     24.30
NL1     24.60
SE1     24.80
SER1    24.85
A1      24.90
AUS1    25.00
DK1     25.05
TS1     25.20
NO1     25.30
UKR1    25.30
Name: average_age, dtype: float64

Overall median fallback: 25.9


In [45]:
mask_age = clubs['average_age'].isna()
clubs.loc[mask_age, 'average_age'] = (
    clubs.loc[mask_age, 'domestic_competition_id']
    .map(median_age_by_comp)
    .fillna(overall_age_median)
)

print(f'Remaining nulls for average_age: {clubs["average_age"].isna().sum()}')
print(f'Range of average_age after cleaning: {clubs["average_age"].min():.1f} - {clubs["average_age"].max():.1f}')

Remaining nulls for average_age: 0
Range of average_age after cleaning: 18.3 - 30.9


**Squad size outlier values handling**

In [46]:
print(f'squad_size distribution:')
print(clubs['squad_size'].describe())

zero_squads = clubs[(clubs['squad_size'] < 11) | (clubs['squad_size'] > 40)]
print(f'\nOutlier clubs with squad_size < 11 or > 40: {len(zero_squads)}')
if len(zero_squads) > 0:
    display(zero_squads)

squad_size distribution:
count    796.000000
mean      26.464824
std        7.650216
min        0.000000
25%       25.000000
50%       28.000000
75%       30.000000
max       52.000000
Name: squad_size, dtype: float64

Outlier clubs with squad_size < 11 or > 40: 57


,club_id,club_code,name,domestic_competition_id,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,last_season,net_transfer_record_eur
58,11126,mordovia-saransk,Mordovia Saransk (-2020),RU1,0,25.70,0,NaN,0,Mordovia Arena,44442,2015,0.0
68,11380,scm-gloria-buzau,ASFC Buzau (2016-2025),RO1,0,26.65,0,NaN,0,Gloria,12336,2024,0.0
82,118373,auckland-fc,Auckland Football Club,AUS1,47,25.20,17,36.2,7,Mount Smart Stadium,25000,2025,0.0
100,12438,volga-nizhniy-novgorod,Volga Nizhniy Novgorod (- 2016),RU1,0,25.70,0,NaN,0,Zentralstadion Lokomotiv,17856,2013,0.0
110,128,ao-xanthi,AO Xanthi,GR1,0,26.65,0,NaN,0,Xanthi Arena,7361,2019,0.0
130,1411,raec-mons,RAEC Mons (- 2015),BE1,0,24.30,0,NaN,0,Stade Charles Tondreau,12662,2013,0.0
158,1506,kardemir-karabukspor,Kardemir Karabükspor,TR1,0,25.80,0,NaN,0,Dr. Necmettin Şeyhoğlu,12400,2017,0.0
169,16239,desna-chernigiv,Desna Chernigiv,UKR1,0,25.30,0,NaN,0,Stadion im. Yuriya Gagarina,12060,2021,0.0
170,16245,fk-sevastopol,FK Sevastopol (- 2014),UKR1,0,25.30,0,NaN,0,SK Sevastopol,5826,2013,0.0
171,16247,pfk-stal-kamyanske,PFK Stal Kamyanske (-2018),UKR1,0,25.30,0,NaN,0,Obolon,5100,2017,0.0


In [47]:
clubs.loc[clubs['squad_size'] < 11 , 'squad_size'] = np.nan
clubs.loc[clubs['squad_size'] > 40 , 'squad_size'] = np.nan
print(f'Nulls for squad_size after cleaning: {clubs["squad_size"].isna().sum()}')

median_squad_by_comp = (
    clubs.dropna(subset=['squad_size'])
    .groupby('domestic_competition_id')['squad_size']
    .median()
)
print('\nMedian squad_size per competition (sample):')
print(median_squad_by_comp.sort_values().head(10))

# fallback
overall_squad_median = clubs['squad_size'].median()
print(f'\nOverall median fallback: {overall_squad_median}')

Nulls for squad_size after cleaning: 57

Median squad_size per competition (sample):
domestic_competition_id
SE1     25.0
DK1     25.5
GB1     26.0
SC1     26.0
COL1    26.0
ES1     26.0
NO1     26.0
NL1     27.0
MEX1    27.0
RU1     27.0
Name: squad_size, dtype: float64

Overall median fallback: 28.0


In [48]:
mask_squad = clubs['squad_size'].isna()
clubs.loc[mask_squad, 'squad_size'] = (
    clubs.loc[mask_squad, 'domestic_competition_id']
    .map(median_squad_by_comp)
    .fillna(overall_squad_median)
)

print(f'Remaining nulls for squad_size: {clubs["squad_size"].isna().sum()}')
print(f'Range of squad_size after cleaning: {clubs["squad_size"].min():.0f} - {clubs["squad_size"].max():.0f}')

Remaining nulls for squad_size: 0
Range of squad_size after cleaning: 12 - 40


**Logical consistency checks for number of players**

In [49]:
invalid_ntp = clubs[clubs['national_team_players'] > clubs['squad_size']]

print(f'Rows where national_team_players > squad_size: {len(invalid_ntp)}')
if len(invalid_ntp) > 0:
    display(invalid_ntp)
    clubs.loc[invalid_ntp.index, 'national_team_players'] = clubs.loc[invalid_ntp.index, 'squad_size']

Rows where national_team_players > squad_size: 0


In [50]:
invalid_foreigners = clubs[clubs['foreigners_number'] > clubs['squad_size']]

print(f'Rows where foreigners_number > squad_size: {len(invalid_foreigners)}')

if len(invalid_foreigners) > 0:
    display(invalid_foreigners)
    clubs.loc[invalid_foreigners.index, 'foreigners_number'] = clubs.loc[invalid_foreigners.index, 'squad_size']
    clubs.loc[invalid_foreigners.index, 'foreigners_percentage'] = 100.0

Rows where foreigners_number > squad_size: 0


**Foreigners percentage null handling**

In [51]:
print(f'Null count for foreigners_percentage before inference: {clubs["foreigners_percentage"].isna().sum()}')

# logical check: ensure foreigners_percentage is between 0 and 100
invalid_foreigners = clubs[
    (clubs['foreigners_percentage'] < 0) | (clubs['foreigners_percentage'] > 100)
]
print(f'\nClubs with invalid foreigners_percentage (<0 or >100): {len(invalid_foreigners)}')
if len(invalid_foreigners) > 0:
    display(invalid_foreigners)

# logical check: foreigners_percentage = (foreigners_number / squad_size) * 100
invalid_percentage = clubs[
    clubs['foreigners_number'].notna() &
    clubs['squad_size'].notna() &
    clubs['foreigners_percentage'].notna() &
    ~np.isclose(
        clubs['foreigners_percentage'],
        (clubs['foreigners_number'] / clubs['squad_size']) * 100,
        atol=1.0
    )
]
print(f'\nClubs where foreigners_percentage does not match foreigners_number/squad_size: {len(invalid_percentage)}')
if len(invalid_percentage) > 0:
    display(invalid_percentage)

Null count for foreigners_percentage before inference: 57

Clubs with invalid foreigners_percentage (<0 or >100): 0

Clubs where foreigners_percentage does not match foreigners_number/squad_size: 16


,club_id,club_code,name,domestic_competition_id,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,last_season,net_transfer_record_eur
47,1095,es-troyes-ac,ESTAC Troyes,FR1,29.0,23.1,16,57.1,5,Stade de l'Aube,21877,2022,0.0
82,118373,auckland-fc,Auckland Football Club,AUS1,30.0,25.2,17,36.2,7,Mount Smart Stadium,25000,2025,0.0
85,1186,torpedo-moskau,Torpedo Moscow,RU1,28.0,25.6,6,23.1,3,Luzhniki Stadium,81000,2022,-2050000.0
205,19789,yeni-malatyaspor,Yeni Malatyaspor,TR1,28.0,22.9,1,10.0,0,Yeni Malatya Stadyumu,25745,2021,778000.0
225,21459,gangwon-fc,Gangwon Football Club,RSK1,34.0,23.7,2,4.7,4,Gangneung High1 Arena,21416,2025,-388000.0
274,24245,umraniyespor,Ümraniyespor,TR1,12.0,25.8,3,30.0,1,Ümraniye Belediyesi Şehir Stadı,3513,2022,0.0
289,2503,boavista-porto-fc,Boavista FC,PO1,28.0,26.8,2,22.2,0,Estádio do Bessa Século XXI,28263,2024,100000.0
337,2861,kv-oostende,KV Oostende,BE1,26.0,23.3,14,50.0,2,Diaz Arena,8432,2022,0.0
356,2995,fc-pacos-de-ferreira,FC Paços de Ferreira,PO1,32.0,25.5,17,50.0,1,Estádio Capital do Móvel,9076,2022,0.0
418,3558,gfc-ajaccio,GFC Ajaccio,FR1,27.0,26.8,1,20.0,1,Stade Ange-Casanova,4050,2015,0.0


In [52]:
computed_percentage = (clubs['foreigners_number'] / clubs['squad_size']) * 100

fix_condition = (
    clubs['foreigners_number'].notna() &
    clubs['squad_size'].notna() &
    (clubs['squad_size'] != 0) &
    (
        clubs['foreigners_percentage'].isna() |
        ~np.isclose(
            clubs['foreigners_percentage'],
            computed_percentage,
            atol=1.0
        )
    )
)

clubs.loc[fix_condition, 'foreigners_percentage'] = computed_percentage[fix_condition]

clubs['foreigners_percentage'] = clubs['foreigners_percentage'].round(2)

print(f'Null count for foreigners_percentage after inference: {clubs["foreigners_percentage"].isna().sum()}')

Null count for foreigners_percentage after inference: 0


**Last season valid range check**

In [53]:
invalid_season = clubs[~clubs['last_season'].between(2010, 2026)]
print(f'Invalid last_season rows (outside 2010-2026): {len(invalid_season)}')
if len(invalid_season) > 0:
    display(invalid_season)

Invalid last_season rows (outside 2010-2026): 0


### **Transfers Table**

**Datetime columns formatting**

In [54]:
transfers["transfer_date"] = pd.to_datetime(transfers["transfer_date"], errors="coerce")

# sanity check
print(f'Transfer date range: {transfers["transfer_date"].min()} to {transfers["transfer_date"].max()}')

Transfer date range: 1993-07-01 00:00:00 to 2030-06-30 00:00:00


**Numeric columns formatting**

In [55]:
numeric_cols = ['transfer_fee', 'market_value_in_eur']
for col in numeric_cols:
    transfers[col] = pd.to_numeric(transfers[col], errors='coerce')

**Transfer season logical check**

In [56]:
def infer_season(date):
    if pd.isna(date):
        return np.nan
    year = date.year
    if date.month >= 7:
        return f"{year % 100:02d}/{(year + 1) % 100:02d}"
    else:
        return f"{(year - 1) % 100:02d}/{year % 100:02d}"

def get_start_year(season):
    if pd.isna(season):
        return np.nan
    try:
        start = int(str(season).split('/')[0])
        if start <= 30:
            return 2000 + start
        else:
            return 1900 + start
    except:
        return np.nan

transfers['inferred_season'] = transfers['transfer_date'].apply(infer_season)

transfers['season_start'] = transfers['transfer_season'].apply(get_start_year)
transfers['inferred_start'] = transfers['inferred_season'].apply(get_start_year)

invalid_season = transfers[
    transfers['season_start'].notna() &
    transfers['inferred_start'].notna() &
    (transfers['season_start'] > transfers['inferred_start'])
]

print(f'Number of records where transfer_season is after inferred season: {len(invalid_season)}')
if len(invalid_season) > 0:
    display(invalid_season)

Number of records where transfer_season is after inferred season: 576


,player_id,transfer_date,transfer_season,from_club_id,to_club_id,from_club_name,to_club_name,transfer_fee,market_value_in_eur,player_name,inferred_season,season_start,inferred_start
517,30321,2026-06-30,26/27,367,515,Rayo Vallecano,Without Club,NaN,400000.0,Óscar Trejo,25/26,2026,2025
565,217115,2026-06-30,26/27,368,244,Sevilla FC,Marseille,0.0,4000000.0,Neal Maupay,25/26,2026,2025
936,531065,2026-06-30,26/27,290,1025,AJ Auxerre,Bologna,0.0,1800000.0,Oussama El Azzouzi,25/26,2026,2025
1022,582982,2026-06-30,26/27,10093,679,Indep. Medellín,Athletico-PR,0.0,500000.0,Hayen Palacios,25/26,2026,2025
1081,622023,2026-06-30,26/27,938,48,FC Thun,Karlsruher SC,0.0,300000.0,Noah Rupp,25/26,2026,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...
155300,114422,2007-01-01,07/08,21078,7750,SSV Ulm U17,SSV Ulm U19,NaN,NaN,Günay Güvenç,06/07,2007,2006
156453,66953,2004-07-29,08/09,11502,141,Galatasaray U21,Galatasaray,NaN,NaN,Murat Akça,04/05,2008,2004
156898,15956,2002-07-01,03/04,28867,8519,Sevilla FC U19,Sevilla B,NaN,NaN,Jesús Navas,02/03,2003,2002
156975,16756,2002-01-01,02/03,14301,6189,Akademia FCSM,Spartak II,NaN,NaN,Oleg Ivanov,01/02,2002,2001


**Logical check for transfer clubs**

In [57]:
self_transfers = transfers[
    (transfers['from_club_id'] == transfers['to_club_id'])
]

print(f'Self-transfer rows (from_club_id == to_club_id): {len(self_transfers)}')
if len(self_transfers) > 0:
    display(self_transfers)
    before = len(transfers)
    transfers = transfers.drop(index=self_transfers.index).reset_index(drop=True)

Self-transfer rows (from_club_id == to_club_id): 7


,player_id,transfer_date,transfer_season,from_club_id,to_club_id,from_club_name,to_club_name,transfer_fee,market_value_in_eur,player_name,inferred_season,season_start,inferred_start
20327,1149569,2025-01-01,24/25,14831,14831,Boca Juniors II,Boca Juniors II,NaN,NaN,Julián Ceballos,24/25,2024,2024
96519,688605,2019-07-01,19/20,42366,42366,FC Basel Yth.,FC Basel Yth.,NaN,NaN,Tim Pfeiffer,19/20,2019,2019
96520,688614,2019-07-01,19/20,42366,42366,FC Basel Yth.,FC Basel Yth.,NaN,NaN,Noah Streit,19/20,2019,2019
124798,1293517,2016-07-01,16/17,75,75,Unknown,Unknown,0.0,NaN,Tim,16/17,2016,2016
138463,349585,2014-01-01,13/14,22380,22380,ABC U20,ABC U20,NaN,NaN,Ayrton Lucas,13/14,2013,2013
153083,528025,2009-01-01,08/09,24067,24067,School Team,School Team,NaN,NaN,Won-cheol Choe,08/09,2008,2008
156256,111455,2005-07-01,05/06,42366,42366,FC Basel Yth.,FC Basel Yth.,NaN,NaN,Granit Xhaka,05/06,2005,2005


**Checking player name against ID**

In [58]:
player_id_to_name = players.set_index('player_id')['name']
valid_player_ids = set(players['player_id'])

In [59]:
unmatched_ids = transfers.loc[
    transfers['player_id'].notna() &
    ~transfers['player_id'].isin(valid_player_ids),
    'player_id'
].unique()

print(f'\nUnique player_id values not found in players table:')
unmatched_ids = sorted(unmatched_ids)
print(unmatched_ids)
print(f'Number of unmatched player_id values: {len(unmatched_ids)}')

transfers['player_name'] = transfers['player_name'].where(
    ~transfers['player_id'].isin(unmatched_ids), 'Unknown'
)

transfers['player_id'] = transfers['player_id'].where(
    ~transfers['player_id'].isin(unmatched_ids), np.nan
)


Unique player_id values not found in players table:
[]
Number of unmatched player_id values: 0


In [60]:
expected_names = transfers['player_id'].map(player_id_to_name)

invalid_name = transfers[
    transfers['player_id'].notna() &
    transfers['player_name'].notna() &
    ~transfers.apply(
        lambda row: (
            str(row['player_name']).lower() in str(player_id_to_name.get(row['player_id'], '')).lower()
            or
            str(player_id_to_name.get(row['player_id'], '')).lower() in str(row['player_name']).lower()
        ),
        axis=1
    )
]

print(f'Rows where player_name does not match player_id: {len(invalid_name)}')

if len(invalid_name) > 0:
    display(invalid_name[['player_id', 'player_name']])

transfers.loc[
    transfers['player_id'].notna(),
    'player_name'
] = expected_names

Rows where player_name does not match player_id: 0


In [61]:
remaining_invalid = transfers.loc[
    transfers['player_id'].notna() &
    ~transfers['player_id'].isin(valid_player_ids)
]


print(f'Remaining invalid player_id: {len(remaining_invalid)}')

Remaining invalid player_id: 0


**Checking club name against ID**

In [62]:
club_id_to_name = clubs.set_index('club_id')['name']
valid_club_ids = set(clubs['club_id'])

In [63]:
# from club check
unmatched_from_ids = transfers.loc[
    transfers['from_club_id'].notna() &
    ~transfers['from_club_id'].isin(valid_club_ids),
    'from_club_id'
].unique()

print(f'\nUnique from_club_id values not found in clubs table:')
unmatched_from_ids = sorted(unmatched_from_ids)
print(unmatched_from_ids)
print(f'Number of unmatched from_club_id values: {len(unmatched_from_ids)}')

transfers['from_club_name'] = transfers['from_club_name'].where(
    ~transfers['from_club_id'].isin(unmatched_from_ids), 'Unknown'
)

transfers['from_club_id'] = transfers['from_club_id'].where(
    ~transfers['from_club_id'].isin(unmatched_from_ids), np.nan
)



Unique from_club_id values not found in clubs table:
[np.int64(1), np.int64(2), np.int64(7), np.int64(8), np.int64(9), np.int64(17), np.int64(21), np.int64(22), np.int64(25), np.int64(28), np.int64(30), np.int64(34), np.int64(45), np.int64(48), np.int64(50), np.int64(52), np.int64(56), np.int64(57), np.int64(63), np.int64(64), np.int64(66), np.int64(68), np.int64(69), np.int64(71), np.int64(72), np.int64(73), np.int64(75), np.int64(77), np.int64(78), np.int64(81), np.int64(83), np.int64(84), np.int64(85), np.int64(87), np.int64(90), np.int64(91), np.int64(92), np.int64(93), np.int64(94), np.int64(95), np.int64(96), np.int64(97), np.int64(101), np.int64(102), np.int64(103), np.int64(104), np.int64(107), np.int64(108), np.int64(109), np.int64(110), np.int64(111), np.int64(113), np.int64(115), np.int64(116), np.int64(119), np.int64(123), np.int64(129), np.int64(135), np.int64(139), np.int64(143), np.int64(151), np.int64(156), np.int64(161), np.int64(163), np.int64(164), np.int64(166), np

In [64]:
# to club check
unmatched_to_ids = transfers.loc[
    transfers['to_club_id'].notna() &
    ~transfers['to_club_id'].isin(valid_club_ids),
    'to_club_id'
].unique()

print(f'\nUnique to_club_id values not found in clubs table:')
unmatched_to_ids = sorted(unmatched_to_ids)
print(unmatched_to_ids)
print(f'Number of unmatched to_club_id values: {len(unmatched_to_ids)}')

transfers['to_club_name'] = transfers['to_club_name'].where(
    ~transfers['to_club_id'].isin(unmatched_to_ids), 'Unknown'
)

transfers['to_club_id'] = transfers['to_club_id'].where(
    ~transfers['to_club_id'].isin(unmatched_to_ids), np.nan
)



Unique to_club_id values not found in clubs table:
[np.int64(1), np.int64(2), np.int64(7), np.int64(8), np.int64(9), np.int64(17), np.int64(21), np.int64(22), np.int64(25), np.int64(28), np.int64(30), np.int64(34), np.int64(45), np.int64(48), np.int64(50), np.int64(52), np.int64(56), np.int64(57), np.int64(63), np.int64(64), np.int64(66), np.int64(68), np.int64(69), np.int64(71), np.int64(72), np.int64(73), np.int64(75), np.int64(77), np.int64(78), np.int64(81), np.int64(83), np.int64(84), np.int64(85), np.int64(87), np.int64(90), np.int64(91), np.int64(92), np.int64(93), np.int64(94), np.int64(95), np.int64(96), np.int64(97), np.int64(101), np.int64(102), np.int64(103), np.int64(104), np.int64(107), np.int64(108), np.int64(109), np.int64(110), np.int64(111), np.int64(113), np.int64(115), np.int64(116), np.int64(119), np.int64(123), np.int64(129), np.int64(135), np.int64(139), np.int64(143), np.int64(151), np.int64(156), np.int64(161), np.int64(163), np.int64(164), np.int64(166), np.i

In [65]:
# from club name check
expected_from_names = transfers['from_club_id'].map(club_id_to_name)

invalid_from_name = transfers[
    transfers['from_club_id'].notna() &
    transfers['from_club_name'].notna() &
    ~transfers.apply(
        lambda row: (
            str(row['from_club_name']).lower() in str(club_id_to_name.get(row['from_club_id'], '')).lower()
            or
            str(club_id_to_name.get(row['from_club_id'], '')).lower() in str(row['from_club_name']).lower()
        ),
        axis=1
    )
]

print(f'Rows where from_club_name does not match club_id: {len(invalid_from_name)}')

if len(invalid_from_name) > 0:
    display(invalid_from_name[['from_club_id', 'from_club_name']])

transfers.loc[
    transfers['from_club_id'].notna(),
    'from_club_name'
] = expected_from_names

Rows where from_club_name does not match club_id: 19445


,from_club_id,from_club_name
4,4467.0,TSV Hartberg
5,242.0,FC Winterthur
7,5546.0,Mlada Boleslav
23,114.0,Besiktas
35,3876.0,Mirassol-SP
...,...,...
156668,159.0,Red Star
156746,159.0,Red Star
156979,43.0,Heart of Midl.
157062,276.0,Hellas Verona


In [66]:
# to club name check
expected_to_names = transfers['to_club_id'].map(club_id_to_name)

invalid_to_name = transfers[
    transfers['to_club_id'].notna() &
    transfers['to_club_name'].notna() &
    ~transfers.apply(
        lambda row: (
            str(row['to_club_name']).lower() in str(club_id_to_name.get(row['to_club_id'], '')).lower()
            or
            str(club_id_to_name.get(row['to_club_id'], '')).lower() in str(row['to_club_name']).lower()
        ),
        axis=1
    )
]

print(f'Rows where to_club_name does not match club_id: {len(invalid_to_name)}')

if len(invalid_to_name) > 0:
    display(invalid_to_name[['to_club_id', 'to_club_name']])
    
transfers.loc[
    transfers['to_club_id'].notna(),
    'to_club_name'
] = expected_to_names

Rows where to_club_name does not match club_id: 25949


,to_club_id,to_club_name
3,30925.0,Gwangju FC
4,14.0,Austria Vienna
7,377.0,Banik Ostrava
8,683.0,Olympiacos
9,3535.0,Ulsan HD
...,...,...
157000,159.0,Red Star
157053,276.0,Hellas Verona
157105,43.0,Heart of Midl.
157114,276.0,Hellas Verona


In [67]:
remaining_from_invalid = transfers.loc[
    transfers['from_club_id'].notna() &
    ~transfers['from_club_id'].isin(valid_club_ids)
]

remaining_to_invalid = transfers.loc[
    transfers['to_club_id'].notna() &
    ~transfers['to_club_id'].isin(valid_club_ids)
]

print(f'\nRemaining invalid from_club_id: {len(remaining_from_invalid)}')
print(f'Remaining invalid to_club_id: {len(remaining_to_invalid)}')


Remaining invalid from_club_id: 0
Remaining invalid to_club_id: 0


**Transfer fee null value handling**

In [68]:
# for null tranfer fees, we create flag fee_disclosed = 0 if transfer_fee is null, else 1
transfers.loc[transfers["transfer_fee"].isna(), "fee_disclosed"] = pd.notnull

transfers["fee_disclosed"] = transfers["transfer_fee"].notna().astype(int)
disclosed_counts = transfers["fee_disclosed"].value_counts().reset_index()
disclosed_counts.columns = ["fee_disclosed", "count"]
print(f"{'\nFee Disclosed':<15} {'Count':>10}")
for _, row in disclosed_counts.iterrows():
    print(f"{row['fee_disclosed']:<15} {row['count']:>10,}")


Fee Disclosed       Count
1                  102,352
0                   54,827


**Logical check between transfer fee and market value**

In [69]:
fee_mv_check = transfers[
    transfers['transfer_fee'].notna() &
    transfers['market_value_in_eur'].notna() &
    (transfers['market_value_in_eur'] > 0) &
    (transfers['transfer_fee'] > 0)
].copy()

fee_mv_check['fee_to_mv_ratio'] = fee_mv_check['transfer_fee'] / fee_mv_check['market_value_in_eur']

extreme_premium = fee_mv_check[fee_mv_check['fee_to_mv_ratio'] > 5]
print(f'Transfers where fee > 5x market value: {len(extreme_premium)}')
if len(extreme_premium) > 0:
    display(
        extreme_premium[['player_id', 'player_name', 'transfer_fee', 'market_value_in_eur', 'fee_to_mv_ratio']]
        .sort_values('fee_to_mv_ratio', ascending=False)
        .head(15)
    )

transfers['extreme_fee_flag'] = 0
if len(extreme_premium) > 0:
    transfers.loc[extreme_premium.index, 'extreme_fee_flag'] = 1
    print(f'Flagged {len(extreme_premium)} rows with extreme_fee_flag = 1')

Transfers where fee > 5x market value: 604


,player_id,player_name,transfer_fee,market_value_in_eur,fee_to_mv_ratio
99766,476344,Emerson Royal,12000000.0,50000.0,240.000000
91384,673492,Morato,7600000.0,50000.0,152.000000
49426,1064871,Joaquín Panichelli,3300000.0,25000.0,132.000000
99520,476344,Emerson Royal,6000000.0,50000.0,120.000000
151631,95424,Kyle Walker,5900000.0,50000.0,118.000000
100113,578539,Chris Richards,5300000.0,50000.0,106.000000
88466,631923,Khalid Al-Ghannam,4850000.0,50000.0,97.000000
81558,565093,Bart Verbruggen,2300000.0,25000.0,92.000000
149507,153427,Bebé,8800000.0,100000.0,88.000000
129555,245078,Nemanja Radonjic,4000000.0,50000.0,80.000000


Flagged 604 rows with extreme_fee_flag = 1


### **Player Valuations Table**

**Logical reference check for player ID**

In [70]:
valid_player_ids = set(players['player_id'])


unmatched_ids = valuations.loc[
    valuations['player_id'].notna() &
    ~valuations['player_id'].isin(valid_player_ids),
    'player_id'
].unique()

print(f'\nUnique player_id values not found in players table:')
unmatched_ids = sorted(unmatched_ids)
print(unmatched_ids)
print(f'Number of unmatched player_id values: {len(unmatched_ids)}')

valuations['player_id'] = valuations['player_id'].where(
    ~valuations['player_id'].isin(unmatched_ids), np.nan
)


Unique player_id values not found in players table:
[np.int64(53180), np.int64(54330), np.int64(74071), np.int64(131074), np.int64(145821), np.int64(156009), np.int64(186744), np.int64(195745), np.int64(208828), np.int64(242580), np.int64(257528), np.int64(262016), np.int64(274963), np.int64(281246), np.int64(287255), np.int64(295086), np.int64(308642), np.int64(311182), np.int64(315511), np.int64(317824), np.int64(344312), np.int64(344421), np.int64(366213), np.int64(367250), np.int64(384664), np.int64(385614), np.int64(396645), np.int64(400121), np.int64(401340), np.int64(412610), np.int64(414926), np.int64(426763), np.int64(448266), np.int64(450290), np.int64(451040), np.int64(451164), np.int64(452447), np.int64(457774), np.int64(469855), np.int64(492999), np.int64(494670), np.int64(503781), np.int64(504103), np.int64(510898), np.int64(515654), np.int64(536653), np.int64(540305), np.int64(550978), np.int64(552948), np.int64(552960), np.int64(569268), np.int64(572817), np.int64(5792

**Logical reference check for club ID**

In [71]:
valid_club_ids = set(clubs['club_id'])

unmatched_ids = valuations.loc[
    valuations['current_club_id'].notna() &
    ~valuations['current_club_id'].isin(valid_club_ids),
    'current_club_id'
].unique()

print(f'\nUnique club_id values not found in clubs table:')
unmatched_ids = sorted(unmatched_ids)
print(unmatched_ids)
print(f'Number of unmatched club_id values: {len(unmatched_ids)}')

valuations['current_club_id'] = valuations['current_club_id'].where(
    ~valuations['current_club_id'].isin(unmatched_ids), np.nan
)


Unique club_id values not found in clubs table:
[np.float64(1.0), np.float64(2.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(17.0), np.float64(21.0), np.float64(22.0), np.float64(25.0), np.float64(28.0), np.float64(30.0), np.float64(34.0), np.float64(45.0), np.float64(48.0), np.float64(50.0), np.float64(52.0), np.float64(56.0), np.float64(57.0), np.float64(63.0), np.float64(64.0), np.float64(66.0), np.float64(69.0), np.float64(71.0), np.float64(72.0), np.float64(73.0), np.float64(75.0), np.float64(77.0), np.float64(78.0), np.float64(81.0), np.float64(83.0), np.float64(84.0), np.float64(85.0), np.float64(87.0), np.float64(90.0), np.float64(91.0), np.float64(92.0), np.float64(93.0), np.float64(94.0), np.float64(95.0), np.float64(96.0), np.float64(97.0), np.float64(101.0), np.float64(102.0), np.float64(103.0), np.float64(104.0), np.float64(107.0), np.float64(108.0), np.float64(109.0), np.float64(110.0), np.float64(111.0), np.float64(113.0), np.float64(115.0), np.float

**Datetime column formatting**

In [72]:
valuations["date"] = pd.to_datetime(valuations["date"], errors="coerce").dt.date

### **Appearances Table**

**Datetime column formatting**

In [73]:
appearances["date"] = pd.to_datetime(appearances["date"], errors="coerce").dt.date

**Numeric columns formatting**

In [74]:
numeric_cols = ['assists', 'goals', 'minutes_played', 'red_cards', 'yellow_cards']
for col in numeric_cols: 
    appearances[col] = pd.to_numeric(appearances[col], errors='coerce')

**Outlier value handling for minutes played**

In [75]:
print(f'minutes_played distribution:')
print(appearances['minutes_played'].describe())

outliers = appearances[appearances['minutes_played'] > 120]
print(f'\nOutlier appearances with minutes_played > 120: {len(outliers)}')
if len(outliers) > 0:
    display(outliers)

mask_minutes = appearances['minutes_played'] > 120
appearances.loc[mask_minutes, 'minutes_played'] = 120

print(f'Range of minutes_played after cleaning: {appearances["minutes_played"].min():.0f} - {appearances["minutes_played"].max():.0f}')

minutes_played distribution:
count    1.862208e+06
mean     6.863815e+01
std      3.015031e+01
min      1.000000e+00
25%      4.500000e+01
50%      9.000000e+01
75%      9.000000e+01
max      1.480000e+02
Name: minutes_played, dtype: float64

Outlier appearances with minutes_played > 120: 3


,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
746780,2901276_296366,2901276,296366,2477,53646,2018-02-21,Oleksiy Gutsulyak,UKR1,0,0,0,0,135
746787,2901276_354145,2901276,354145,2477,614,2018-02-21,Jorge Carrascal,UKR1,0,0,1,0,135
1655860,4500194_422466,4500194,422466,1244,265,2024-12-04,Javi Hernández,CDR,1,0,1,0,148


Range of minutes_played after cleaning: 1 - 120


**Logical reference check for player name against ID**

In [76]:
player_id_to_name = players.set_index('player_id')['name']
valid_player_ids = set(players['player_id'])

In [77]:
unmatched_ids = appearances.loc[
    appearances['player_id'].notna() &
    ~appearances['player_id'].isin(valid_player_ids),
    'player_id'
].unique()

print(f'\nUnique player_id values not found in players table:')
unmatched_ids = sorted(unmatched_ids)
print(unmatched_ids)
print(f'Number of unmatched player_id values: {len(unmatched_ids)}')

appearances['player_name'] = appearances['player_name'].where(
    ~appearances['player_id'].isin(unmatched_ids), 'Unknown'
)

appearances['player_id'] = appearances['player_id'].where(
    ~appearances['player_id'].isin(unmatched_ids), np.nan
)

unmatched_ids = appearances.loc[
    appearances['player_id'].notna() &
    ~appearances['player_id'].isin(valid_player_ids),
    'player_id'
].unique()

print(f'\nUnique player_id values not found in players table after cleaning:')
unmatched_ids = sorted(unmatched_ids)
print(unmatched_ids)
print(f'Number of unmatched player_id values after cleaning: {len(unmatched_ids)}')


Unique player_id values not found in players table:
[np.int64(380365)]
Number of unmatched player_id values: 1

Unique player_id values not found in players table after cleaning:
[]
Number of unmatched player_id values after cleaning: 0


In [78]:
expected_names = appearances['player_id'].map(player_id_to_name)

invalid_name = appearances[
    appearances['player_id'].notna() &
    appearances['player_name'].notna() &
    ~appearances.apply(
        lambda row: (
            str(row['player_name']).lower() in str(player_id_to_name.get(row['player_id'], '')).lower()
            or
            str(player_id_to_name.get(row['player_id'], '')).lower() in str(row['player_name']).lower()
        ),
        axis=1
    )
]

print(f'Rows where player_name does not match player_id: {len(invalid_name)}')

if len(invalid_name) > 0:
    display(invalid_name[['player_id', 'player_name']])

appearances.loc[
    appearances['player_id'].notna(),
    'player_name'
] = expected_names

Rows where player_name does not match player_id: 0


In [79]:
remaining_invalid = appearances.loc[
    appearances['player_id'].notna() &
    ~appearances['player_id'].isin(valid_player_ids)
]


print(f'Remaining invalid player_id: {len(remaining_invalid)}')

Remaining invalid player_id: 0


**Logical reference check for club ID columns**

In [80]:
# player_club_id check
valid_club_ids = set(clubs['club_id'])

unmatched_ids = appearances.loc[
    appearances['player_club_id'].notna() &
    ~appearances['player_club_id'].isin(valid_club_ids),
    'player_club_id'
].unique()

print(f'\nUnique club_id values not found in clubs table:')
unmatched_ids = sorted(unmatched_ids)
print(unmatched_ids)
print(f'Number of unmatched club_id values: {len(unmatched_ids)}')

appearances['player_club_id'] = appearances['player_club_id'].where(
    ~appearances['player_club_id'].isin(unmatched_ids), np.nan
)

unmatched_ids = appearances.loc[
    appearances['player_club_id'].notna() &
    ~appearances['player_club_id'].isin(valid_club_ids),
    'player_club_id'
].unique()

print(f'\nUnique club_id values not found in clubs table after cleaning:')
unmatched_ids = sorted(unmatched_ids)
print(unmatched_ids)
print(f'Number of unmatched club_id values after cleaning: {len(unmatched_ids)}')


Unique club_id values not found in clubs table:
[np.int64(1), np.int64(2), np.int64(7), np.int64(9), np.int64(22), np.int64(25), np.int64(30), np.int64(48), np.int64(52), np.int64(56), np.int64(64), np.int64(69), np.int64(72), np.int64(77), np.int64(78), np.int64(81), np.int64(84), np.int64(94), np.int64(95), np.int64(107), np.int64(108), np.int64(109), np.int64(119), np.int64(129), np.int64(156), np.int64(163), np.int64(164), np.int64(171), np.int64(187), np.int64(208), np.int64(220), np.int64(253), np.int64(254), np.int64(271), np.int64(279), np.int64(282), np.int64(293), np.int64(318), np.int64(332), np.int64(337), np.int64(349), np.int64(352), np.int64(355), np.int64(358), np.int64(362), np.int64(365), np.int64(384), np.int64(404), np.int64(408), np.int64(420), np.int64(429), np.int64(433), np.int64(440), np.int64(466), np.int64(500), np.int64(540), np.int64(588), np.int64(602), np.int64(630), np.int64(648), np.int64(663), np.int64(666), np.int64(698), np.int64(699), np.int64(704)

In [81]:
# player_current_club_id check
valid_club_ids = set(clubs['club_id'])

unmatched_ids = appearances.loc[
    appearances['player_current_club_id'].notna() &
    ~appearances['player_current_club_id'].isin(valid_club_ids),
    'player_current_club_id'
].unique()

print(f'\nUnique club_id values not found in clubs table:')
unmatched_ids = sorted(unmatched_ids)
print(unmatched_ids)
print(f'Number of unmatched club_id values: {len(unmatched_ids)}')

appearances['player_current_club_id'] = appearances['player_current_club_id'].where(
    ~appearances['player_current_club_id'].isin(unmatched_ids), np.nan
)

unmatched_ids = appearances.loc[
    appearances['player_current_club_id'].notna() &
    ~appearances['player_current_club_id'].isin(valid_club_ids),
    'player_current_club_id'
].unique()

print(f'\nUnique club_id values not found in clubs table after cleaning:')
unmatched_ids = sorted(unmatched_ids)
print(unmatched_ids)
print(f'Number of unmatched club_id values after cleaning: {len(unmatched_ids)}')


Unique club_id values not found in clubs table:
[np.int64(-1), np.int64(2), np.int64(7), np.int64(22), np.int64(45), np.int64(48), np.int64(64), np.int64(77), np.int64(84), np.int64(85), np.int64(91), np.int64(119), np.int64(123), np.int64(156), np.int64(161), np.int64(164), np.int64(184), np.int64(208), np.int64(220), np.int64(253), np.int64(279), np.int64(282), np.int64(332), np.int64(337), np.int64(349), np.int64(352), np.int64(355), np.int64(358), np.int64(440), np.int64(466), np.int64(500), np.int64(515), np.int64(518), np.int64(540), np.int64(588), np.int64(602), np.int64(630), np.int64(634), np.int64(656), np.int64(663), np.int64(698), np.int64(708), np.int64(710), np.int64(786), np.int64(790), np.int64(810), np.int64(829), np.int64(840), np.int64(947), np.int64(965), np.int64(967), np.int64(988), np.int64(990), np.int64(991), np.int64(1002), np.int64(1008), np.int64(1017), np.int64(1020), np.int64(1028), np.int64(1034), np.int64(1035), np.int64(1037), np.int64(1045), np.int64(

## **Saving Cleaned Dataset**

In [82]:
players.to_csv(os.path.join(OUTPUT_DIR, 'cleaned_players.csv'), index=False)
clubs.to_csv(os.path.join(OUTPUT_DIR, 'cleaned_clubs.csv'), index=False)
transfers.to_csv(os.path.join(OUTPUT_DIR, 'cleaned_transfers.csv'), index=False)
valuations.to_csv(os.path.join(OUTPUT_DIR, 'cleaned_valuations.csv'), index=False)    
appearances.to_csv(os.path.join(OUTPUT_DIR, 'cleaned_appearances.csv'), index=False)
